###Setup

In [1]:
import urllib.request
import os
import re

url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
file_path = "the-verdict.txt"

In [2]:
if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)

with open(file_path, 'r', encoding='utf-8') as f1, open(file_path, 'r', encoding='utf-8') as f2:
    lines = f1.readlines()
    raw_text = f2.read()

###Simple Preprocess

In [7]:
# \s matches whitespace (spaces, tabs and new lines)
# () denotes a capturing group
# [] Square brackets to create a matching list that will match on any one of
# -- the characters in the list (only one).
# So this regex matches either comma or periods (only one of them), OR newlines
# Multiple stages to show the different types of splitting that we can do
def simple_preprocess(text: str, stage:int, keep_whitespaces:bool=False) -> str:
    if str(stage).isnumeric and len(str(stage)) == 1 and isinstance(stage, int):
        if stage == 1:
            # Only commas, periods, and whitespaces
            res = re.split(r'([,.]|\s)', text)
        elif stage == 2:
            # Comma, dot, colon, semicolon, question mark, underscore,
            # -- and exclamation point
            res = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        if not keep_whitespaces:
            res = [txt.strip() for txt in res if txt.strip()]
        return res
    else:
        return text

In [10]:
n = 50
print(f"Total amount of characters: {len(raw_text)} ")
print(f"Total amount of lines: {len(lines)}")
print(f"First line: {lines[0]}")
print(f"First {n} characters: {raw_text[:n]}")

preprocessed = simple_preprocess(raw_text, 2, False)
print(f"First {n} characters of the modified text: {preprocessed[:n]}")
print(f"Lenght: {len(preprocessed)}")

Total amount of characters: 20479 
Total amount of lines: 165
First line: I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

First 50 characters: I HAD always thought Jack Gisburn rather a cheap g
First 50 characters of the modified text: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself']
Lenght: 4690


In [11]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(f"Vocab size: {vocab_size}")

vocab = {token:id for id, token in enumerate(all_words)}
for idx, item in enumerate(vocab.items()):
    print(item)
    if idx >= 50:
        break

Vocab size: 1130
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


### Simple Tokenizer Class

In [12]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        # Vocab contains the dictionary mapping each word to its id
        self.str_to_int = vocab
        # Reverse mapping, from token id to the respective token
        self.int_to_str = {idx: tok for tok, idx in vocab.items()}

    def encode(self, text, keep_whitespaces: bool=False):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        if not keep_whitespaces:
            preprocessed = [
                item.strip() for item in preprocessed if item.strip()
            ]
        ids = [self.str_to_int[tok] for tok in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
# Preprocess first to get a vocabulary size
preprocessed_text = simple_preprocess(raw_text, 2, False)
all_tokens = sorted(set(preprocessed))
vocab = {token:id for id, token in enumerate(all_tokens)}
print(len(vocab.items()))

In [13]:
SimpleTok = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride."""
ids = SimpleTok.encode(text)
print(f"Encoded text: {ids}")
print(f"Decoded ids: {SimpleTok.decode(ids)}")

1130
Encoded text: [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
Decoded ids: " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


If we use a text such as this, we will get a `KeyError` "`cakes` is not present in the current vocabulary."

In [23]:
text ="""I like waffles"""
try:
    ids = SimpleTok.encode(text)
    print(ids)
    print(SimpleTok.decode(ids))
except KeyError as e:
    print(f"KeyError: the {e} word is not present in the current vocabulary.")

KeyError: the 'waffles' word is not present in the current vocabulary.


Therefore we will add <|unk|> and <|endoftext|> tokens

In [20]:
preprocessed_text = simple_preprocess(raw_text, 2, False)
all_tokens = sorted(list(set(preprocessed_text)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
extended_vocab = {token:id for id, token in enumerate(all_tokens)}
print(len(extended_vocab.items()))

# Last 5 elements
for key, value in enumerate(list(extended_vocab.items())[-5:]):
    print(key, value)

1132
0 ('younger', 1127)
1 ('your', 1128)
2 ('yourself', 1129)
3 ('<|endoftext|>', 1130)
4 ('<|unk|>', 1131)


####Update the Tokenizer Class

In [39]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        # Vocab contains the dictionary mapping each word to its id
        self.str_to_int:dict = vocab
        # Reverse mapping, from token id to the respective token
        self.int_to_str:dict = {idx: tok for tok, idx in vocab.items()}

    def encode(self, text, keep_whitespaces: bool=False):
        preprocessed:list = re.split(r'([,.?_!"()\']|--|\s)', text)
        if not keep_whitespaces:
            preprocessed = [
                item.strip() for item in preprocessed if item.strip()
            ]
        # Check on the vocabulary keys (words)
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item
                        in preprocessed]
        ids = [self.str_to_int[tok] for tok in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

Trying it out

In [40]:
SimpleTokV2 = SimpleTokenizerV2(extended_vocab)

In [41]:
text1 = """I like waffles."""
text2 = """And I like ice cream!"""
text = "<|endoftext|>".join((text1, text2))
print(text)

I like waffles.<|endoftext|>And I like ice cream!


In [42]:
ids = SimpleTokV2.encode(text)
print(ids)
print(SimpleTokV2.decode(ids))

[53, 628, 1131, 7, 1131, 53, 628, 1131, 1131, 0]
I like <|unk|>. <|unk|> I like <|unk|> <|unk|>!


###Byte Pair Encoding

In [44]:
%pip install tiktoken
import tiktoken
print(f"Version: {tiktoken.__version__}")

Version: 0.12.0


In [45]:
tokenizer = tiktoken.get_encoding("gpt2")

In [47]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace"
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(f"Tokens: {integers}")

Tokens: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271]


In [48]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace


In [52]:
eof_tok = "<|endoftext|>"
print(f"the <|endoftext|> token is associated with the id {tokenizer.encode(eof_tok, allowed_special={"<|endoftext|>"})}")

the <|endoftext|> token is associated with the id [50256]
